# **AI Engineer Assistant:**
**Agent with LLM + VectorDB** |
**Agent with LLM + Memory**

In [1]:
# Install packages by using requirements.txt
%pip install -r "/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap12/requirements.txt"

In [2]:
# # Installing package: watermark
# !pip install -q watermark

In [3]:
# # Installing package: langchain
# !pip install -q langchain

In [4]:
# # Installing package: langchain-openai
# !pip install -q langchain-openai

In [5]:
# # Installing package: chromadb
# !pip install -q chromadb

In [6]:
# # Installing package: sqlalchemy (langchain dependency)
# !pip install -q sqlalchemy

In [7]:
# Imports
import os
import textwrap
import chromadb
import langchain
import sqlalchemy
import langchain_openai
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.chains import SequentialChain
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
# from langchain.document_loaders import WebBaseLoader
from langchain.indexes import VectorstoreIndexCreator
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
# from langchain.chat_models import ChatOpenAI
import string

# new
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.chat_models import ChatOpenAI

import warnings
warnings.filterwarnings('ignore')

## Defining the LLM to work with

In [8]:
# Check if .env file is present
dotenv_path = '/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap12/env'

if os.path.exists(dotenv_path):
    print(f"The .env file exists at: {dotenv_path}")
else:
    print(f"Error: The .env file does NOT exist at: {dotenv_path}")
    print("Please ensure the path is correct and the file is present.")

The .env file exists at: /content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap12/env


In [9]:
# Check if API KEY is present
from dotenv import load_dotenv

dotenv_path = '/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap12/env'
load_success = load_dotenv(dotenv_path)

print(f"load_dotenv() returned: {load_success}")

if 'OPENAI_API_KEY' in os.environ:
    print('OPENAI_API_KEY is now available in the environment.')
else:
    print('OPENAI_API_KEY is NOT available in the environment. Please check your .env file content.')

load_dotenv() returned: True
OPENAI_API_KEY is now available in the environment.


In [10]:
# Defining the LLM
openai_llm = OpenAI(temperature=0.5)

In [11]:
# Testing prompt
prompt_result = openai_llm.invoke("What is Google's mission?")

In [12]:
# See result
print(prompt_result)



Google's mission is to organize the world's information and make it universally accessible and useful.


## Working with PromptTemplates

In [13]:
# Defining tech template
tech_prompt_template_name = PromptTemplate(
    input_variables = ["tech"],
    template = "I want to know more about {tech} in the context of AI field. Tell me an overview about it"
)

In [14]:
# Generating specific tech prompt for LLM
langgraph_prompt_template = tech_prompt_template_name.format(tech = "langgraph")
print(langgraph_prompt_template)

I want to know more about langgraph in the context of AI field. Tell me an overview about it


## Working with Simple Chains

In [15]:
# Creating the Chain
tech_overview_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_name)

In [16]:
# Invoke the Chain
print(tech_overview_chain.invoke("langgraph"))

{'tech': 'langgraph', 'text': '\n\nLanggraph, also known as language graph, is a concept in the field of artificial intelligence (AI) that refers to the representation of language and its relationships in a graphical structure. It is a way of organizing and visualizing the complex interconnections between words, phrases, and concepts in a language.\n\nThe idea of langgraph is based on the understanding that language is not just a linear sequence of words, but a complex network of interconnected elements. This network can be represented as a graph, where nodes represent individual words or concepts, and edges represent the relationships between them.\n\nOne of the main applications of langgraph is in natural language processing (NLP), which is a subfield of AI that deals with the understanding and generation of human language by computers. By using langgraph, NLP systems can better analyze and interpret the meaning of words and sentences, as well as identify patterns and relationships b

In [17]:
# Creating the Chain with Verbose
tech_overview_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_name, verbose = True)

In [18]:
# Invoke the Chain
print(tech_overview_chain.invoke("langgraph"))



> Entering new LLMChain chain...
Prompt after formatting:
I want to know more about langgraph in the context of AI field. Tell me an overview about it

> Finished chain.
{'tech': 'langgraph', 'text': '\n\nLanggraph, also known as language graph, is a concept in the field of artificial intelligence (AI) that refers to the representation of language in a graphical form. It is a visual representation of the relationships between words and concepts in a language, and it is used to improve natural language processing (NLP) and other AI applications.\n\nThe idea behind langgraph is to create a structured and interconnected network of words and their meanings, similar to how the human brain processes language. This allows AI systems to understand and interpret language in a more human-like manner.\n\nLanggraph is built upon the principles of graph theory, which is a mathematical concept that represents objects and their relationships using nodes (representing objects) and edges (representin

In [19]:
# Create now a Simple Chain that uses more than 1
# Creating another prompt template
tech_prompt_template_use_cases = PromptTemplate(
    input_variables = ["tech_overview"],
    template = "Based on that {tech_overview}, list the 3 most used use cases"
)


In [20]:
# Creating the dual chain
tech_uses_cases_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_use_cases, verbose = True)

In [21]:
# Putting the 2 chains together
overall_tech_chain = SimpleSequentialChain(chains = [tech_overview_chain, tech_uses_cases_chain], verbose = True)

In [22]:
# Invoke the final chain
print(overall_tech_chain.invoke("langgraph"))



> Entering new SimpleSequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
I want to know more about langgraph in the context of AI field. Tell me an overview about it

> Finished chain.


Langgraph, or language graph, is a concept in the field of artificial intelligence (AI) that refers to a graphical representation of language and its underlying structure. It is a way to represent the complex relationships between words, phrases, and sentences in a language.

The idea of langgraph is based on the fact that language is not just a collection of words, but a highly structured and interconnected system. Each word has its own meaning, but it also has relationships with other words, such as synonyms, antonyms, and hyponyms. These relationships form a network or graph, which can be visualized as a langgraph.

Langgraph is often used in natural language processing (NLP) and computational linguistics to analyze and understand the structure of language. By mapp

## Working with Sequential Chains

In [23]:
# Defining tech template
tech_prompt_template_name = PromptTemplate(
    input_variables = ["tech"],
    template = "I want to know more about {tech} in the context of AI field. Tell me an overview about it in one phrase"
)

In [24]:
# Creating the Chain
tech_overview_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_name, output_key = "tech_overview")

In [25]:
# Creating another prompt template for chain 2
tech_prompt_template_use_cases = PromptTemplate(
    input_variables = ["tech_overview"],
    template = "Based on that {tech_overview}, list the 3 most used use cases"
)


In [26]:
# Creating the chain 2
tech_uses_cases_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_use_cases, verbose = True, output_key = "tech_use_cases")

In [27]:
# Creating the Sequential Chain
overall_tech_chain = SequentialChain(
    chains = [tech_overview_chain, tech_uses_cases_chain],
    input_variables = ["tech"],
    output_variables = ["tech_overview", "tech_use_cases"],
    verbose = True
)

In [28]:
# Invoke the final chain
response = overall_tech_chain.invoke({"tech": "langchain"})
print(response)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Based on that 

"Langchain is an AI-powered language processing platform that enables seamless communication and understanding across different languages.", list the 3 most used use cases

> Finished chain.

> Finished chain.
{'tech': 'langchain', 'tech_overview': '\n\n"Langchain is an AI-powered language processing platform that enables seamless communication and understanding across different languages."', 'tech_use_cases': '\n\n1. Multilingual Communication: Langchain can be used for real-time translation and interpretation, allowing individuals and businesses to communicate effectively with people who speak different languages.\n\n2. Language Learning: Langchain can be used as a tool for language learning, providing users with accurate translations, pronunciation assistance, and other language learning resources.\n\n3. Global Business: Langchain can help businesses expand their glo

In [29]:
# Format the response
response_formated = f" Tech Stack: {response['tech']}\n\n Teck Overview: {response['tech_overview']} \n\n Use Cases: {response['tech_use_cases']}"
print(response_formated)

 Tech Stack: langchain

 Teck Overview: 

"Langchain is an AI-powered language processing platform that enables seamless communication and understanding across different languages." 

 Use Cases: 

1. Multilingual Communication: Langchain can be used for real-time translation and interpretation, allowing individuals and businesses to communicate effectively with people who speak different languages.

2. Language Learning: Langchain can be used as a tool for language learning, providing users with accurate translations, pronunciation assistance, and other language learning resources.

3. Global Business: Langchain can help businesses expand their global reach by providing accurate and efficient translation services for their products, services, and communications with clients and partners from different countries. 


## Working with Memory

In [30]:
# Defining a Chain
tech_overview_chain = LLMChain(llm = openai_llm, prompt = tech_prompt_template_name)

In [31]:
# Invoke Chain
overview = tech_overview_chain.invoke(
    {"tech": "langchain"}
)
print(overview)

{'tech': 'langchain', 'text': '\n\n"Langchain is a decentralized platform that combines AI and blockchain technology for secure and efficient language data processing and analysis."'}


In [32]:
# See memory?
type(tech_overview_chain.memory)

NoneType

In [33]:
# Creating a Memory Object for the Chain
tech_memory = ConversationBufferMemory()

In [34]:
# Creating a new Chain with the Memory now
tech_overview_chain_with_memory = LLMChain(llm = openai_llm, prompt = tech_prompt_template_name, memory = tech_memory)

In [35]:
# Use the Chain
result1 = tech_overview_chain_with_memory.invoke({"tech": "langchain"})
print(result1)

{'tech': 'langchain', 'history': '', 'text': '\n\n"Langchain is a blockchain-based platform that utilizes artificial intelligence to facilitate secure and efficient cross-language communication and data exchange."'}


In [36]:
# Use the Chain (2)
result2 = tech_overview_chain_with_memory.invoke({"tech": "langgraph"})
print(result2)

{'tech': 'langgraph', 'history': 'Human: langchain\nAI: \n\n"Langchain is a blockchain-based platform that utilizes artificial intelligence to facilitate secure and efficient cross-language communication and data exchange."', 'text': '\n\n"Langgraph is a graph-based representation of natural language used in AI for text analysis and understanding."'}


In [37]:
# See memory history
print(tech_overview_chain_with_memory.memory.buffer)

Human: langchain
AI: 

"Langchain is a blockchain-based platform that utilizes artificial intelligence to facilitate secure and efficient cross-language communication and data exchange."
Human: langgraph
AI: 

"Langgraph is a graph-based representation of natural language used in AI for text analysis and understanding."


## Working with ConversationChain

In [38]:
# Creating the conversational object
tech_conversation = ConversationChain(llm = openai_llm)

In [39]:
# Print conversation template
print(tech_conversation.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [40]:
# Invoke the object
result1 = tech_conversation.invoke(input = "I want to know more about langchain in the context of AI field. Tell me an overview about it")
print(result1)

{'input': 'I want to know more about langchain in the context of AI field. Tell me an overview about it', 'history': '', 'response': ' Langchain is a relatively new concept in the AI field that combines natural language processing and blockchain technology. It aims to create a decentralized platform for natural language data processing and analysis. This means that instead of relying on a central authority or server, the data is stored and processed on a distributed network of computers, making it more secure and transparent. Langchain has the potential to revolutionize the way AI systems handle and understand human language, as it can provide a more accurate and efficient way of processing large amounts of data. It also has the potential to improve privacy and data ownership for individuals, as they can control their own data on the blockchain. However, langchain is still in its early stages and there are many challenges to overcome before it can be fully implemented in the AI field.'

In [41]:
# Invoke the object (2)
result2 = tech_conversation.invoke(input = "I want to know more about langgraph in the context of AI field. Tell me an overview about it")
print(result2)

{'input': 'I want to know more about langgraph in the context of AI field. Tell me an overview about it', 'history': 'Human: I want to know more about langchain in the context of AI field. Tell me an overview about it\nAI:  Langchain is a relatively new concept in the AI field that combines natural language processing and blockchain technology. It aims to create a decentralized platform for natural language data processing and analysis. This means that instead of relying on a central authority or server, the data is stored and processed on a distributed network of computers, making it more secure and transparent. Langchain has the potential to revolutionize the way AI systems handle and understand human language, as it can provide a more accurate and efficient way of processing large amounts of data. It also has the potential to improve privacy and data ownership for individuals, as they can control their own data on the blockchain. However, langchain is still in its early stages and t

In [42]:
# See conversation memory
print(tech_conversation.memory.buffer)

Human: I want to know more about langchain in the context of AI field. Tell me an overview about it
AI:  Langchain is a relatively new concept in the AI field that combines natural language processing and blockchain technology. It aims to create a decentralized platform for natural language data processing and analysis. This means that instead of relying on a central authority or server, the data is stored and processed on a distributed network of computers, making it more secure and transparent. Langchain has the potential to revolutionize the way AI systems handle and understand human language, as it can provide a more accurate and efficient way of processing large amounts of data. It also has the potential to improve privacy and data ownership for individuals, as they can control their own data on the blockchain. However, langchain is still in its early stages and there are many challenges to overcome before it can be fully implemented in the AI field.
Human: I want to know more abo

## Working with ConversationBufferWindowMemory in ConversationChain

In [43]:
# Defining a ConversationBufferWindowMemory object
tech_conversation_window = ConversationBufferWindowMemory(k = 1)

In [44]:
# Create the conversation
tech_conversation_window_chain = ConversationChain(llm = openai_llm, memory = tech_conversation_window)

In [45]:
# Invoke the conversation
result1 = tech_conversation_window_chain.invoke(input = "I want to know more about langchain in the context of AI field. Tell me an overview about it")
print(result1)

{'input': 'I want to know more about langchain in the context of AI field. Tell me an overview about it', 'history': '', 'response': " Langchain is a decentralized platform that uses artificial intelligence to analyze and translate languages in real time. It is specifically designed for use in the AI field, as it allows for seamless communication and collaboration between researchers and developers from different linguistic backgrounds. Langchain uses advanced deep learning algorithms to accurately translate complex technical jargon and terminology, making it an invaluable tool for AI professionals. Additionally, Langchain's decentralized nature ensures security and privacy, as all data is encrypted and stored on a distributed network. Overall, Langchain is revolutionizing the way language barriers are overcome in the AI field, promoting global collaboration and innovation. Is there anything else you would like to know?"}


In [46]:
# Invoke the conversation (2)
result2 = tech_conversation_window_chain.invoke(input = "I want to know more about langgraph in the context of AI field. Tell me an overview about it")
print(result2)

{'input': 'I want to know more about langgraph in the context of AI field. Tell me an overview about it', 'history': "Human: I want to know more about langchain in the context of AI field. Tell me an overview about it\nAI:  Langchain is a decentralized platform that uses artificial intelligence to analyze and translate languages in real time. It is specifically designed for use in the AI field, as it allows for seamless communication and collaboration between researchers and developers from different linguistic backgrounds. Langchain uses advanced deep learning algorithms to accurately translate complex technical jargon and terminology, making it an invaluable tool for AI professionals. Additionally, Langchain's decentralized nature ensures security and privacy, as all data is encrypted and stored on a distributed network. Overall, Langchain is revolutionizing the way language barriers are overcome in the AI field, promoting global collaboration and innovation. Is there anything else y

In [47]:
# Invoke the conversation (3)
result3 = tech_conversation_window_chain.invoke(input = "I want to know more about Python in the context of AI field. Tell me an overview about it")
print(result3)

{'input': 'I want to know more about Python in the context of AI field. Tell me an overview about it', 'history': 'Human: I want to know more about langgraph in the context of AI field. Tell me an overview about it\nAI:  I\'m sorry, I do not have any information on a platform called "langgraph." Could you provide more context or clarify the name?', 'response': ' Sure! Python is a high-level, interpreted programming language that is widely used in the field of AI. It was first released in 1991 and has since become one of the most popular languages for AI development. Its simple syntax and powerful libraries make it a great choice for tasks such as data analysis, machine learning, and natural language processing. Many popular AI frameworks, such as TensorFlow and PyTorch, are built on top of Python. Additionally, its large and active community provides a wealth of resources and support for AI projects. Overall, Python is a crucial tool for AI developers and is constantly evolving to meet

In [48]:
# Print memory buffer
print(tech_conversation_window_chain.memory.buffer)

Human: I want to know more about Python in the context of AI field. Tell me an overview about it
AI:  Sure! Python is a high-level, interpreted programming language that is widely used in the field of AI. It was first released in 1991 and has since become one of the most popular languages for AI development. Its simple syntax and powerful libraries make it a great choice for tasks such as data analysis, machine learning, and natural language processing. Many popular AI frameworks, such as TensorFlow and PyTorch, are built on top of Python. Additionally, its large and active community provides a wealth of resources and support for AI projects. Overall, Python is a crucial tool for AI developers and is constantly evolving to meet the needs of the field.


## 1º - AI Enginner ChatBot Working with LangChain, VectorDB and Web Scraping

In [49]:
# Extracting Web Data
tech_data = WebBaseLoader(
    "https://blog.dsacademy.com.br/langchain-aplicacoes-de-inteligencia-artificial-generativa-com-llms/"
)

In [50]:
# Load the documents
documents = tech_data.load()
len(documents)

1

In [51]:
# See the only document
document = documents[0]
print(document)

page_content='





 


LangChain - Aplicações de Inteligência Artificial Generativa com LLMs - Data Science Academy






























































































 
Pular para o conteúdo



Data Science Academy
Sua carreira elevada a outro nível







Home
Deep Learning Book
Blog
Todos os Cursos
Login
 









LangChain – Aplicações de Inteligência Artificial Generativa com LLMs









 




abril 28, 2025

LangChain Large Language Models (LLMs)


LangChain – Aplicações de Inteligência Artificial Generativa com LLMs







Hoje em dia, as apps precisam ser superinteligentes na hora de entender a linguagem e é aí que entra o LangChain! Ele facilita a conexão de modelos de IA (Inteligência Artificial) com todos os tipos de fontes de dados diferentes, para que você possa obter soluções de Processamento de Linguagem Natural (PLN) totalmente personalizadas.
LangChain é um framework de código aberto (open-source) criado para simplificar o 

In [52]:
# See document keys (to filter what we want to see)
document.__dict__.keys()

dict_keys(['id', 'metadata', 'page_content', 'type'])

In [53]:
# See pagent content
document.page_content[:1000]

'\n\n\n\n\n\n \n\n\nLangChain - Aplicações de Inteligência Artificial Generativa com LLMs - Data Science Academy\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \nPular para o conteúdo\n\n\n\nData Science Academy\nSua carreira elevada a outro nível\n\n\n\n\n\n\n\nHome\nDeep Learning Book\nBlog\nTodos os Cursos\nLogin\n \n\n\n\n\n\n\n\n\n\nLangChain – Aplicações de Inteligência Artificial Generativa com LLMs\n\n\n\n\n\n\n\n\n\n \n\n\n\n\nabril 28, 2025\n\nLangChain Large Language Models (LLMs)\n\n\nLangChain – Aplicações de Inteligência Artificial Generativa com LLMs\n\n\n\n\n\n\n\nHoje em dia, as apps precisam ser superinteligentes na hora de entender a linguagem e é aí que entra o LangChain! Ele facilita a conexão de modelos de IA (Inteligência Artificial) com todos os tipos de fontes de dados diferentes, para que você possa obter soluções de Pr

In [54]:
# See document metadata
document.metadata

{'source': 'https://blog.dsacademy.com.br/langchain-aplicacoes-de-inteligencia-artificial-generativa-com-llms/',
 'title': 'LangChain - Aplicações de Inteligência Artificial Generativa com LLMs - Data Science Academy',
 'description': 'LangChain é um framework de código aberto (open-source) criado para simplificar o desenvolvimento de aplicativos usando Large Language Models (LLMs), como aqueles disponibilizados por OpenAI ou Hugging Face.',
 'language': 'pt-BR'}

In [68]:
# Creating the template
tech_template = """ This is a conversation between a AI Engineer working in a company and an Expert Senior AI Engineer.
You are the Senior AI Engineer with experience in the tech LangChain, LLMs, LangGrapgh, Knowledge and Semantic Layers and Agentic Architecture.
{context}

Answer always with precision and give answers well structured.

Question: {question}
Answer:

"""

In [69]:
# Creating the PromptTemplate
prompt = PromptTemplate(
    input_variables = ["context", "question"],
    template = tech_template
)

In [70]:
# See the format
print(prompt.format(context = document.page_content[:100], question = "What is langchain?"))

 This is a conversation between a AI Engineer working in a company and an Expert Senior AI Engineer.
You are the Senior AI Engineer with experience in the tech LangChain, LLMs, LangGrapgh, Knowledge and Semantic Layers and Agentic Architecture.






 


LangChain - Aplicações de Inteligência Artificial Generativa com LLMs - Data Science Acade

Answer always with precision and give answers well structured.

Question: What is langchain?
Answer:




In [71]:
# Creating the embeddings
embeddings = OpenAIEmbeddings()

In [59]:
# Creating the VectorDB
tech_db = Chroma.from_documents(documents, embeddings)

In [73]:
# Creating the RetrievalQA
chain_type_kwargs = {"prompt": prompt}
openai_llm = ChatOpenAI(temperature = 0)
chain = RetrievalQA.from_chain_type(
          llm = openai_llm,
          chain_type = "stuff",
          retriever = tech_db.as_retriever(),
          chain_type_kwargs = chain_type_kwargs
        )

In [74]:
# Defining query
query = "What is langchain?"

In [75]:
# Invoke the chain - Simple prompt
answer = chain.run(query)
answer

'LangChain is a powerful open-source framework designed to facilitate the development of applications powered by a language model, particularly a large language model (LLM). It goes beyond standard API calls by being data-aware and agentic, allowing connections to various data sources for richer and more personalized experiences. It also empowers a language model to dynamically interact with its environment. LangChain streamlines the development of various applications, such as chatbots, Generative Question-Answering (GQA), and summarization, by "chaining" components from various modules to create unique applications built around an LLM.'

In [64]:
# Creating the ChatAgent
while True:
  prompt = input(prompt = "AI Engineer: ")
  print()

  if prompt.strip().lower().strip(string.punctuation + " ") == "thanks":
    break

  result = chain.run(prompt)
  ai_engineer_agent_response = textwrap.dedent(result)
  print(f"AI Engineer Agent: {ai_engineer_agent_response}")
  print()

AI Engineer: What is langchain?

AI Engineer Agent: LangChain is an open-source framework designed to simplify the development of applications using Large Language Models (LLMs). It allows for the connection of AI models with various data sources, enabling the creation of personalized Natural Language Processing (NLP) solutions. It also empowers LLMs to dynamically interact with their environment.

AI Engineer: What is main used for?

AI Engineer Agent: Main is used as the entry point for the execution of a program in most programming languages. It is where the program starts and typically contains the main logic and function calls to other parts of the program. Without a main function, the program would not know where to begin and would not be able to execute. 

AI Engineer: What are the attributes of LangChain?

AI Engineer Agent: 
As a Senior AI Engineer with experience in LangChain, LLMs, LangGraph, Knowledge and Semantic Layers, and Agentic Architecture, I can provide some insight

## 2º - AI Enginner ChatBot Working with LangChain and ConversationBufferMemory

In [76]:
# Defining the LLM
openai_llm = ChatOpenAI(temperature = 0)

In [77]:
# Creating the template
tech_template = """ This is a conversation between a AI Engineer working in a company and an Expert Senior AI Engineer.
You are the Expert Senior AI Engineer with experience in the tech LangChain, LLMs, LangGrapgh, Knowledge and Semantic Layers and Agentic Architecture.

Answer always with precision and give answers well structured.

Current conversation:
{history}

AI Engineer: {input}
Expert Senior AI Engineer:

"""

In [78]:
# Creating the prompt template
prompt = PromptTemplate(
    input_variables = ["history", "input"],
    template = tech_template
)

In [79]:
# Creating the Conversation Chain with Memory
ConversationBufferMemory_chain = ConversationChain(
    llm = openai_llm,
    prompt = prompt,
    verbose = False,
    memory = ConversationBufferMemory(human_prefix = "AI Engineer", ai_prefix = "Expert Senior AI Engineer")
)
#

In [80]:
# Creating the ChatAgent
while True:
  prompt = input(prompt = "AI Engineer: ")
  print()

  if prompt.strip().lower().strip(string.punctuation + " ") == "thanks":
    break

  result = ConversationBufferMemory_chain.run(prompt)
  ai_engineer_agent_response = textwrap.dedent(result)
  print(f"AI Engineer Agent: {ai_engineer_agent_response}")
  print()

AI Engineer: What is LangGrapgh in the AI field?

AI Engineer Agent: LangGraph is a framework that focuses on representing and analyzing language data in a graph structure. It allows for the modeling of relationships between words, phrases, and concepts in a more structured and interconnected way. This can help in tasks such as natural language understanding, sentiment analysis, and information retrieval by leveraging the rich semantic information encoded in the graph structure.

AI Engineer: That is normal used iwith LangChain?

AI Engineer Agent: LangGraph can be used in conjunction with LangChain, which is a framework for building and deploying AI models that operate on language data. By integrating LangGraph into LangChain, we can enhance the capabilities of the AI models by leveraging the structured representation of language data provided by LangGraph. This can lead to more accurate and efficient natural language processing tasks, as well as improved performance in tasks such as 